# ShopTalk · Phase 8: Chat UI (Gradio)

Puts a chat interface in front of the Phase 7 inference API, so a shopper can ask for products in plain language, see product cards, ask follow-ups and rate the answers.

| Plan item (Phase 8) | How it is done here |
|---|---|
| 1. Gradio app that calls the API | `src/ui/app.py` calls `POST /recommend/stream` (answer streams in token by token); `POST /recommend` is a fallback (`SHOPTALK_UI_STREAM=0`) |
| 2. Show the answer and product cards (image, title, ID, link) | Chat on the left, product grid on the right (`src/ui/render.py`); citations like `[ID: B082SP22KL]` are highlighted |
| 3. Keep `history_entry` for follow-ups | Each response's `history_entry` is kept per browser tab (last 3 turns) and sent back with the next question |
| 4. Thumbs up / down → `POST /feedback` | 👍 / 👎 buttons plus an optional comment, rated against the latest answer's `request_id` |
| 5. Clicks → `/click/...` | Card links go to the UI's `/go/...` route, which logs the click through the API's `/click` and redirects to the product page |

**Architecture**

```
Browser ──HTTPS──> cloudflared tunnel ──> UI server :7860  (FastAPI + Gradio, src/ui/app.py)
                                              │  /        chat page
                                              │  /go/...  click → API /click → redirect to product
                                              ▼
                                   ShopTalk API :8000 (Phase 7, stays private on 127.0.0.1)
                                              ▼
                                   RAG pipeline (Qwen2.5-3B + LoRA E5 + Chroma), on the GPU
```
Only the UI gets a public URL. The UI holds no model, so the same two processes move to Docker / AWS unchanged in Phase 11.

**Notebook map:** 0 Setup · 1 Config · 2 UI code · 3 Unit tests · 4 Start the API · 5 Start the UI · 6 Smoke tests · 7 UI latency · 8 Public URL & manual test · 9 Logs · 10 Exit checklist · 11 Stop · 12 Write-up

**Runtime:** Colab with a T4 GPU when `API_MODE = "local"` (the default: the API runs in this session, as in Phase 7). `API_MODE = "fake"` needs no GPU (for working on the UI), and `"remote"` points the UI at an API running elsewhere. **Run all cells top to bottom.** Phases 6 and 7 must have been run in this project folder.

## 0. Setup

### 📘 0.1 Google Drive Integration & Verification

*   **Purpose:** Mounts Google Drive at `/content/drive` and checks the mount is live (a plain folder check can pass on the VM disk without Drive).
*   **Expected Results:** Prints `✅ Google Drive is mounted`. Raises an error if mounting fails.
*   **Parameters:** None.

In [ ]:
import os
from google.colab import drive
if not os.path.ismount("/content/drive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount failed, probably because /content/drive already has local files. "
                           "Runtime > Disconnect and delete runtime, then run this cell first.") from e
assert os.path.ismount("/content/drive"), "Google Drive is NOT mounted"
print("✅ Google Drive is mounted")

### 📘 0.2 Library Installation

*   **Purpose:** Installs Gradio 6 for the UI, plus the Phase 7 API packages at the versions verified in Phase 7 (needed only when the API runs in this session). No `-U` / `--force-reinstall`, so Colab's preinstalled torch / transformers stack is left alone (that is what broke Phase 7 at first).
*   **Expected Results:** Quiet install with no errors. If pip reports that it changed an already-imported package, use *Runtime → Restart session* and continue from 0.3.
*   **Parameters:** `gradio>=6,<7`: the UI. `torchao>=0.16`: transformers 5.x refuses to load the LLM with Colab's older torchao.

In [ ]:
!pip install -q "gradio>=6.0,<7" "fastapi>=0.110" "uvicorn[standard]" requests httpx pytest
# Phase 7 API dependencies (same pins as Phase 7's requirements-api.txt)
!pip install -q chromadb==1.5.9 sentence-transformers==5.7.0 peft==0.21.2 "torchao>=0.16"

### 📘 0.3 Hugging Face Token Configuration

*   **Purpose:** Loads `HF_TOKEN` from Colab Secrets and passes it on to the API process (only needed when the API runs here).
*   **Expected Results:** Prints `HF_TOKEN set` or a note that it is not needed.
*   **Parameters:** None.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("No HF_TOKEN secret: fine for the Qwen models")

## 1. Config
Edit this cell only.

### 📘 1.1 Configuration Parameters

*   `REPO_DIR`: the ShopTalk project folder on Drive (same as Phases 6 and 7).
*   `API_MODE`: where the UI gets its answers from.
    *   `"local"` (default): start the Phase 7 API with the real pipeline in this session (needs the T4 GPU; 1–3 min to load).
    *   `"fake"`: start the Phase 7 API with its fake pipeline (`SHOPTALK_FAKE=1`): no GPU, 8 made-up products. Use it to work on the UI; the smoke and latency checks then say little about quality.
    *   `"remote"`: do not start an API; use `REMOTE_API_URL` (e.g. the Phase 7 tunnel URL from another session).
*   `API_PORT` / `UI_PORT`: ports of the API (private) and the UI server.
*   `LOCAL_CHROMA_DIR`: local SSD copy of the Chroma index used by the API (Drive I/O is slow; see Phase 7, 1.2).
*   `START_UI_TUNNEL`: give the UI a temporary public HTTPS URL via cloudflared (for demos and testing from your laptop or phone).
*   `N_UI_LATENCY`: number of chat turns for the UI latency check (section 7).
*   `UI_FIRST_TOKEN_P95_TARGET_MS` / `UI_TOTAL_P95_TARGET_MS`: targets for what the user waits for the first words of the answer and for the full answer.

**Expected results:** prints the mode, the API URL the UI will use, and the Phase 7 latency for comparison. Fails fast if the Phase 7 API code is missing.

In [ ]:
import sys, json, time, subprocess
from pathlib import Path

REPO_DIR    = Path("/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk")    # <- your project folder
CONFIG_PATH = REPO_DIR / "src/rag/rag_config.json"       # written by Phase 6
LOG_DIR     = REPO_DIR / "logs"                          # requests.jsonl, feedback.jsonl (shared with Phase 7)
RESULTS_DIR = REPO_DIR / "results/phase8"
LOCAL_CHROMA_DIR = "/content/chroma_local_api"

API_MODE = "local"            # "local" (real pipeline here, GPU) | "fake" (no GPU) | "remote"
REMOTE_API_URL = ""           # used only when API_MODE == "remote", e.g. "https://xyz.trycloudflare.com"
API_PORT = 8000
UI_PORT = 7860
START_UI_TUNNEL = True
N_UI_LATENCY = 20
UI_FIRST_TOKEN_P95_TARGET_MS = 1500
UI_TOTAL_P95_TARGET_MS = 5000

# Fail fast: Phase 7 must have been run in this folder
need = [REPO_DIR / p for p in ("src/api/main.py", "src/api/service.py", "src/api/schemas.py", "src/api/fake_pipeline.py")]
if API_MODE == "local":
    need += [CONFIG_PATH, REPO_DIR / "src/rag/factory.py"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, "Not found (is Drive mounted? did Phase 7 run in this folder?):\n  " + "\n  ".join(missing)
assert API_MODE in ("local", "fake", "remote"), API_MODE
assert API_MODE != "remote" or REMOTE_API_URL.startswith("http"), "Set REMOTE_API_URL for API_MODE='remote'"

API_URL = REMOTE_API_URL.rstrip("/") if API_MODE == "remote" else f"http://127.0.0.1:{API_PORT}"
UI_BASE = f"http://127.0.0.1:{UI_PORT}"
for d in ("src/ui", "tests"):
    os.makedirs(REPO_DIR / d, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
CHECKS = {}

phase7_csv = REPO_DIR / "results/phase7/api_latency_ms.csv"
print(f"API_MODE = {API_MODE} | UI talks to {API_URL} | UI on port {UI_PORT}")
print("Phase 7 API latency (ms):", "\n" + open(phase7_csv).read() if phase7_csv.exists() else "not found")

### 📘 1.2 Environment Check

*   **Purpose:** Confirms Gradio and the HTTP libraries import in this kernel (and, for local mode, the model libraries).
*   **Expected Results:** `✅ All required libraries are loaded` and the Gradio version (6.x).
*   **Parameters:** None.

In [ ]:
import importlib
libs = ["gradio", "fastapi", "uvicorn", "requests", "httpx", "pytest"]
if API_MODE != "remote":
    libs += ["chromadb", "sentence_transformers", "transformers", "peft"]
missing_libs = []
for lib in libs:
    try:
        importlib.import_module(lib)
    except Exception as e:
        missing_libs.append(f"{lib} ({type(e).__name__})")
import gradio as gr
assert not missing_libs, f"❌ Missing: {missing_libs}. Run 0.2, then Runtime > Restart session."
assert gr.__version__.split(".")[0] == "6", f"Expected Gradio 6.x, found {gr.__version__}. Run 0.2 and restart."
print(f"✅ All required libraries are loaded | gradio {gr.__version__}")

## 2. Write the UI code to `src/ui/`
Same pattern as Phases 6 and 7: these cells write files into the repo, so they are versioned on GitHub and reused in Phase 11 (Docker / AWS).

### 📖 2.1 Overview: how the UI is put together

| File | What it does |
|---|---|
| `src/ui/client.py` | `ShopTalkClient`: the only place that talks HTTP to the API (`health`, `stream`, `recommend`, `feedback`, `click`). Parses Server-Sent Events from raw chunks so tokens appear as soon as they arrive. |
| `src/ui/render.py` | Pure functions that turn API responses into HTML product cards, the chat Markdown and the status lines. Everything from the catalog or the LLM is HTML-escaped. |
| `src/ui/app.py` | The Gradio layout and its handlers, plus a small FastAPI server that serves the UI at `/` and the click redirect at `/go/...`. |
| `tests/test_ui.py` | Tests the handlers, the client and the `/go` route against the Phase 7 API running with its fake pipeline. |

**One chat turn**

1. The user's message appears at once (`add_user_message`).
2. `chat_turn` calls `POST /recommend/stream` with the question, the last 3 `history_entry` values and the tab's `session_id`.
3. `meta` event → the product cards are drawn (≈0.3 s in Phase 7).
4. `token` events → the answer grows in the chat (first words ≈0.8 s).
5. `done` event → the final answer replaces the streamed text (the API removes citations to products that were not shown, so it can differ slightly), and the new `history_entry` is stored for follow-ups.

**Why a `/go` route instead of linking to the API's `/click` directly?** The browser cannot reach `127.0.0.1:8000` on the Colab VM. `/go/{request_id}/{product_id}?rank=N` runs on the UI server, calls the API's `/click` server-side (which writes the click to `feedback.jsonl`), then redirects the browser to the product page. The API never needs a public URL.

**Per-tab state:** `session_id` (`ui-…`, logged with every request, rating and click), the last 3 `history_entry` values, and the latest `request_id` (what 👍 / 👎 refer to). *New chat* clears the history but keeps the session id.

### 📘 2.2 Package Initialization

*   **Purpose:** Marks `src/ui/` as a Python package.
*   **Expected Results:** Writes `src/ui/__init__.py`.
*   **Parameters:** None.

In [ ]:
%%writefile src/ui/__init__.py
"""ShopTalk chat UI (Phase 8)."""

### 📘 2.3 API Client (`client.py`)

*   **Purpose:** Wraps the Phase 7 endpoints. Errors (API down, 503 while models load) become `ApiError` with a readable message, so the UI can show it instead of crashing.
*   **Expected Results:** Writes `src/ui/client.py`.
*   **Parameters:** `base_url`: API address. `timeout`: seconds for a full answer (default 300).

In [ ]:
%%writefile src/ui/client.py
"""HTTP client for the ShopTalk inference API (Phase 7), used by the Phase 8 UI.

Keeps the UI free of HTTP details: the Gradio handlers call these methods and get plain dicts back.
Only `requests` is needed, so the same client works in Colab, in tests and later on AWS (Phase 11).

Endpoints used:
  GET  /health                          -> status banner
  POST /recommend/stream                -> SSE events: meta (products), token (answer text), done, error
  POST /recommend                       -> non-streaming fallback
  POST /feedback                        -> thumbs up / down
  GET  /click/{request_id}/{product_id} -> logs the click, then 307-redirects to the product page
"""
from __future__ import annotations

import json
from typing import Iterator, Optional

import requests


class ApiError(RuntimeError):
    """Raised when the API is unreachable or answers with an error the user should see."""


class ShopTalkClient:
    """
    Thin wrapper around the Phase 7 endpoints.

    Args:
        base_url (str): API address, e.g. "http://127.0.0.1:8000" or a public tunnel URL.
        timeout (float): Seconds to wait for a full response (streaming reads use it per chunk).
    """

    def __init__(self, base_url: str, timeout: float = 300.0):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = requests.Session()

    # ---------------- helpers ----------------
    def _url(self, path: str) -> str:
        return f"{self.base_url}{path}"

    @staticmethod
    def _raise_for(r: requests.Response) -> None:
        """Turns HTTP errors into ApiError with the API's own message (e.g. 503 while loading)."""
        if r.status_code < 400:
            return
        try:
            detail = r.json().get("detail", r.text)
        except ValueError:
            detail = r.text
        raise ApiError(f"API error {r.status_code}: {detail}")

    # ---------------- endpoints ----------------
    def health(self) -> dict:
        """Returns /health, or {'status': 'unreachable', 'error': ...} if the API cannot be reached."""
        try:
            r = self.session.get(self._url("/health"), timeout=10)
            self._raise_for(r)
            return r.json()
        except (requests.RequestException, ApiError) as e:
            return {"status": "unreachable", "error": str(e)}

    def recommend(self, query: str, history: Optional[list] = None, session_id: Optional[str] = None) -> dict:
        """Non-streaming call to POST /recommend."""
        try:
            r = self.session.post(self._url("/recommend"), timeout=self.timeout,
                                  json={"query": query, "history": history or [], "session_id": session_id})
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        self._raise_for(r)
        return r.json()

    def stream(self, query: str, history: Optional[list] = None,
               session_id: Optional[str] = None) -> Iterator[tuple]:
        """
        Calls POST /recommend/stream and yields (event, data) as each Server-Sent Event arrives.

        Events: ("meta", {request_id, products, ...}), ("token", {text}), ("done", {...full response}),
        ("error", {request_id, detail}).
        """
        try:
            r = self.session.post(self._url("/recommend/stream"), stream=True, timeout=(10, self.timeout),
                                  json={"query": query, "history": history or [], "session_id": session_id})
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        with r:
            self._raise_for(r)
            yield from parse_sse(r.iter_content(chunk_size=None))

    def feedback(self, request_id: str, rating: str, comment: Optional[str] = None,
                 session_id: Optional[str] = None) -> dict:
        """POST /feedback with rating 'up' or 'down'."""
        try:
            r = self.session.post(self._url("/feedback"), timeout=10, json={
                "request_id": request_id, "rating": rating, "comment": comment or None, "session_id": session_id})
        except requests.RequestException as e:
            raise ApiError(f"API unreachable: {e}") from e
        self._raise_for(r)
        return r.json()

    def click(self, request_id: str, product_id: str, rank: Optional[int] = None,
              session_id: Optional[str] = None) -> Optional[str]:
        """
        Logs a click via GET /click/... without following the redirect.

        Returns:
            str | None: The product page URL from the redirect, or None if the API did not redirect.
        """
        params = {k: v for k, v in (("rank", rank), ("session_id", session_id)) if v is not None}
        try:
            r = self.session.get(self._url(f"/click/{request_id}/{product_id}"), params=params,
                                 allow_redirects=False, timeout=10)
        except requests.RequestException:
            return None
        return r.headers.get("location") if r.status_code in (301, 302, 303, 307, 308) else None


def parse_sse(chunks) -> Iterator[tuple]:
    """
    Parses a Server-Sent Events byte stream into (event, data) tuples.

    Works on raw chunks (not lines) so tokens are shown as soon as they arrive; requests' iter_lines
    would buffer them. Events are separated by a blank line; `data:` is JSON.
    """
    buf = ""
    for chunk in chunks:
        if not chunk:
            continue
        buf += chunk.decode("utf-8", errors="replace") if isinstance(chunk, bytes) else chunk
        buf = buf.replace("\r\n", "\n")
        while "\n\n" in buf:
            block, buf = buf.split("\n\n", 1)
            event, data_lines = "message", []
            for line in block.split("\n"):
                if line.startswith("event:"):
                    event = line[6:].strip()
                elif line.startswith("data:"):
                    data_lines.append(line[5:].lstrip())
            if data_lines:
                raw = "\n".join(data_lines)
                try:
                    yield event, json.loads(raw)
                except ValueError:
                    yield event, {"text": raw}

### 📘 2.4 Rendering (`render.py`)

*   **Purpose:** Builds the product-card grid (image or 🛍️ placeholder, title, brand · color · type, price, matched-filter chips, ID, *View product* link), formats citations in the answer, and builds the latency and API-status lines.
*   **Expected Results:** Writes `src/ui/render.py`.
*   **Parameters:** `CSS`: card styles, passed to Gradio when the app is mounted.

In [ ]:
%%writefile src/ui/render.py
"""HTML / Markdown rendering for the Phase 8 UI. Pure functions (no Gradio), so they are unit-tested.

Everything that comes from the catalog or the LLM is HTML-escaped before it is shown.
"""
from __future__ import annotations

import html
import re
from typing import Optional
from urllib.parse import quote

ID_RE = re.compile(r"\[ID:\s*([A-Za-z0-9_\-]+)\]")

CSS = """
.st-grid {display:grid; grid-template-columns:repeat(auto-fill,minmax(180px,1fr)); gap:12px;}
.st-card {border:1px solid var(--border-color-primary); border-radius:10px; padding:10px;
          background:var(--background-fill-secondary); display:flex; flex-direction:column; gap:6px;}
.st-img {width:100%; aspect-ratio:1/1; object-fit:contain; border-radius:6px; background:#fff;}
.st-noimg {width:100%; aspect-ratio:1/1; border-radius:6px; display:flex; align-items:center;
           justify-content:center; font-size:42px; background:var(--background-fill-primary);}
.st-title {font-weight:600; font-size:0.92em; line-height:1.25; display:-webkit-box;
           -webkit-line-clamp:3; -webkit-box-orient:vertical; overflow:hidden;}
.st-meta {font-size:0.8em; opacity:0.8;}
.st-price {font-weight:700;}
.st-id {font-family:monospace; font-size:0.78em; opacity:0.75;}
.st-chip {display:inline-block; font-size:0.72em; padding:1px 6px; border-radius:8px; margin-right:4px;
          background:var(--color-accent-soft);}
.st-link {margin-top:auto; font-size:0.85em; font-weight:600;}
.st-note {font-size:0.85em; padding:8px 10px; border-radius:8px; background:var(--background-fill-secondary);
          margin-bottom:10px;}
"""

EMPTY_PRODUCTS = "<div class='st-note'>Product recommendations will appear here.</div>"


def esc(x) -> str:
    return html.escape("" if x is None else str(x), quote=True)


def click_href(link_base: str, request_id: Optional[str], product_id: str, rank: int,
               session_id: Optional[str] = None, fallback: Optional[str] = None) -> str:
    """
    Link used on a product card. It points at the UI's /go route, which logs the click through the
    API's /click endpoint and then redirects to the product page. Without a request_id there is
    nothing to log against, so the direct product link is used.
    """
    if not request_id:
        return fallback or "#"
    href = f"{link_base.rstrip('/')}/go/{quote(request_id)}/{quote(product_id)}?rank={rank}"
    if session_id:
        href += f"&session_id={quote(session_id)}"
    return href


def format_price(p: dict) -> str:
    if p.get("price") is None or p.get("price_unknown"):
        return "price n/a"
    return f"${p['price']:,.2f}"


def product_cards_html(products: list, request_id: Optional[str] = None, link_base: str = "",
                       session_id: Optional[str] = None, unmet: Optional[list] = None,
                       no_match: bool = False) -> str:
    """Renders the product grid shown next to the chat."""
    if no_match or not products:
        return ("<div class='st-note'>No matching products in the catalog. Try a broader description "
                "or relax a filter such as price or color.</div>")
    parts = []
    if unmet:
        parts.append(f"<div class='st-note'>⚠️ None of these has everything you asked for "
                     f"(not matched: {esc(', '.join(unmet))}). These are the closest options.</div>")
    parts.append("<div class='st-grid'>")
    for rank, p in enumerate(products, 1):
        pid = str(p.get("id", ""))
        img = p.get("image_url")
        img_html = (f"<img class='st-img' src='{esc(img)}' alt='{esc(p.get('title'))}' loading='lazy'>"
                    if isinstance(img, str) and img.startswith(("http://", "https://"))
                    else "<div class='st-noimg'>🛍️</div>")
        meta = " · ".join(esc(x) for x in (p.get("brand"), p.get("color"), p.get("product_type")) if x)
        chips = "".join(f"<span class='st-chip'>✓ {esc(k)}</span>"
                        for k, v in (p.get("matched") or {}).items() if v)
        href = click_href(link_base, request_id, pid, rank, session_id, fallback=p.get("link"))
        parts.append(
            f"<div class='st-card'>{img_html}"
            f"<div class='st-title'>{rank}. {esc(p.get('title'))}</div>"
            f"<div class='st-meta'>{meta}</div>"
            f"<div><span class='st-price'>{esc(format_price(p))}</span></div>"
            f"<div>{chips}</div>"
            f"<div class='st-id'>ID: {esc(pid)}</div>"
            f"<a class='st-link' href='{esc(href)}' target='_blank' rel='noopener'>View product ↗</a>"
            f"</div>")
    parts.append("</div>")
    return "".join(parts)


def format_answer(text: str) -> str:
    """Chat answer as Markdown: product citations like [ID: B082SP22KL] are shown in bold code style.
    Angle brackets are escaped so catalog titles or LLM output cannot inject HTML into the chat."""
    text = (text or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return ID_RE.sub(lambda m: f"**`{m.group(1)}`**", text).strip()


def latency_line(latency_ms: Optional[dict], client_ms: Optional[dict] = None) -> str:
    """Small status line under the chat: server-side stage times and what the user actually waited."""
    if not latency_ms:
        return ""
    keys = [k for k in ("parse", "retrieve", "generate", "queue", "total") if latency_ms.get(k) is not None]
    s = " · ".join(f"{k} {latency_ms[k]:,.0f} ms" for k in keys)
    if client_ms:
        s += " | UI: " + " · ".join(f"{k.replace('_ms', '').replace('_', ' ')} {v:,.0f} ms"
                                    for k, v in client_ms.items() if v is not None)
    return f"⏱️ {s}"


def health_banner(h: dict, api_url: str) -> str:
    status = h.get("status")
    if status == "ok":
        return (f"🟢 **API ready** · LLM `{h.get('llm')}` · index {h.get('index_size') or 0:,} products "
                f"· `{api_url}`")
    if status == "loading":
        return f"🟡 **API is loading models** (1–3 min). Refresh the status in a moment · `{api_url}`"
    return f"🔴 **API not available** ({esc(h.get('error') or status)}) · `{api_url}`"

### 📘 2.5 Gradio App & UI Server (`app.py`)

*   **Purpose:** Defines the chat layout and its handlers (`add_user_message`, `chat_turn`, `send_feedback`, `clear_chat`) and `create_server()`, which mounts Gradio at `/` next to the `/go` click redirect and a `/ui-health` check.
*   **Expected Results:** Writes `src/ui/app.py`. Started in section 5 with `python -m src.ui.app`.
*   **Parameters:** Environment: `SHOPTALK_API_URL` (API address), `SHOPTALK_UI_PORT` (default 7860), `SHOPTALK_UI_STREAM=0` to use `/recommend` instead of streaming.

In [ ]:
%%writefile src/ui/app.py
"""ShopTalk chat UI (Phase 8): a Gradio app on top of the Phase 7 inference API.

What the user gets:
  * a chat box: the answer streams in token by token (POST /recommend/stream)
  * product cards next to the chat (image, title, brand, price, ID, "View product" link)
  * follow-ups ("something similar but cheaper"): each response's `history_entry` is kept and sent back
  * 👍 / 👎 (+ optional comment) on the latest answer -> POST /feedback
  * clicks on "View product" go through /go/... on this UI server, which logs them via the API's
    /click endpoint and then redirects to the product page

The UI holds no model: it only talks HTTP to the API, so the API can stay private (127.0.0.1) and
only the UI needs a public URL. The same file runs in Colab and in Docker / AWS (Phase 11).

Run:
    SHOPTALK_API_URL=http://127.0.0.1:8000 python -m src.ui.app        # serves on port 7860
"""
from __future__ import annotations

import os
import re
import time
import uuid
from typing import Optional

import gradio as gr
from fastapi import FastAPI
from fastapi.responses import RedirectResponse

from .client import ApiError, ShopTalkClient
from .render import (CSS, EMPTY_PRODUCTS, format_answer, health_banner, latency_line, product_cards_html)

MAX_HISTORY = 3                      # the API uses the last 3 turns
PRODUCT_URL = "https://www.amazon.com/dp/{id}"
_ID_RE = re.compile(r"^[A-Za-z0-9_\-]{1,64}$")

EXAMPLES = ["Show me a red shirt for men under $50", "Waterproof hiking boots, size 10",
            "A gift for someone who loves coffee", "Something cozy for a small living room",
            "Find me a black leather wallet"]


# ---------------- per-browser-session state ----------------
def new_state(session_id: Optional[str] = None) -> dict:
    """State kept per browser tab: session id, last turns for follow-ups, and the latest request."""
    return {"session_id": session_id or f"ui-{uuid.uuid4().hex[:12]}", "history": [],
            "last_request_id": None, "voted": {}}


def _assistant(chat: list, text: str) -> list:
    """Sets the text of the last assistant message (adds one if the last message is the user's)."""
    if chat and chat[-1].get("role") == "assistant":
        chat[-1] = {"role": "assistant", "content": text}
    else:
        chat.append({"role": "assistant", "content": text})
    return chat


# ---------------- handlers (plain functions, unit-tested without a browser) ----------------
def add_user_message(message: str, chat: Optional[list]):
    """Step 1 of a turn: show the user's message at once and clear the textbox."""
    chat = list(chat or [])
    message = (message or "").strip()
    if message:
        chat.append({"role": "user", "content": message})
    return "", chat, message


def chat_turn(client: ShopTalkClient, message: str, chat: Optional[list], state: Optional[dict],
              link_base: str = "", use_stream: bool = True):
    """
    Step 2 of a turn: call the API and stream the result into the UI.

    Yields (chat, products_html, latency_markdown, state) every time something new arrives:
    first the product cards (meta event), then the growing answer (token events), then the final
    answer (done event; the API cleans citations, so it can differ slightly from the streamed text).
    """
    chat = list(chat or [])
    state = dict(state or new_state())
    if not message:
        yield chat, gr.update(), gr.update(), state
        return
    if len(message) > 500:
        yield _assistant(chat, "That message is too long (500 characters max). Please shorten it."), \
            gr.update(), "", state
        return

    sid, history = state["session_id"], state["history"][-MAX_HISTORY:]
    t0 = time.perf_counter()
    products_html, text = gr.update(), ""
    client_ms = {"products_ms": None, "first_token_ms": None, "total_ms": None}
    done = None
    try:
        if use_stream:
            yield _assistant(chat, "🔎 Searching the catalog…"), gr.update(), "", state
            for event, data in client.stream(message, history=history, session_id=sid):
                if event == "meta":
                    client_ms["products_ms"] = (time.perf_counter() - t0) * 1000
                    state["last_request_id"] = data.get("request_id")
                    products_html = product_cards_html(
                        data.get("products", []), data.get("request_id"), link_base, sid,
                        data.get("unmet_constraints"), data.get("no_match", False))
                    yield _assistant(chat, "✍️ Writing an answer…"), products_html, "", state
                elif event == "token":
                    if client_ms["first_token_ms"] is None:
                        client_ms["first_token_ms"] = (time.perf_counter() - t0) * 1000
                    text += data.get("text", "")
                    yield _assistant(chat, format_answer(text) + " ▌"), products_html, "", state
                elif event == "done":
                    done = data
                elif event == "error":
                    raise ApiError(f"{data.get('detail', 'error')} (request_id={data.get('request_id')})")
        else:
            yield _assistant(chat, "🔎 Searching and writing an answer…"), gr.update(), "", state
            done = client.recommend(message, history=history, session_id=sid)
    except ApiError as e:
        msg = str(e)
        if "503" in msg:
            msg = "The models are still loading on the server. Please try again in a minute."
        yield _assistant(chat, f"⚠️ {msg}"), products_html, "", state
        return

    if done is None:
        yield _assistant(chat, "⚠️ The answer was cut off. Please try again."), products_html, "", state
        return

    client_ms["total_ms"] = (time.perf_counter() - t0) * 1000
    rid = done.get("request_id")
    state["last_request_id"] = rid
    state["history"] = (state["history"] + [done["history_entry"]])[-MAX_HISTORY:]
    products_html = product_cards_html(done.get("products", []), rid, link_base, sid,
                                       done.get("unmet_constraints"), done.get("no_match", False))
    state["last_response"] = {"request_id": rid, "products": done.get("products", []),
                              "no_match": done.get("no_match"), "latency_ms": done.get("latency_ms"),
                              "client_ms": {k: round(v) for k, v in client_ms.items() if v is not None},
                              "answer": done.get("answer", "")}
    yield (_assistant(chat, format_answer(done.get("answer", ""))), products_html,
           latency_line(done.get("latency_ms"), {k: v for k, v in client_ms.items() if v is not None}), state)


def send_feedback(client: ShopTalkClient, rating: str, comment: str, state: Optional[dict]):
    """👍 / 👎 on the latest answer. Returns (status_markdown, comment_box_value, state)."""
    state = dict(state or new_state())
    rid = state.get("last_request_id")
    if not rid:
        return "Ask something first, then rate the answer.", comment, state
    voted = dict(state.get("voted") or {})
    if voted.get(rid) == rating and not (comment or "").strip():
        return "Already recorded, thanks!", comment, state
    try:
        client.feedback(rid, rating, (comment or "").strip()[:1000] or None, state["session_id"])
    except ApiError as e:
        return f"⚠️ Could not send feedback: {e}", comment, state
    voted[rid] = rating
    state["voted"] = voted
    return ("Thanks! 👍 recorded." if rating == "up" else "Thanks, 👎 recorded. It will help us improve."), "", state


def clear_chat(state: Optional[dict]):
    """New conversation: empties the chat and the follow-up history, keeps the session id."""
    s = new_state((state or {}).get("session_id"))
    return [], EMPTY_PRODUCTS, "", "", s


# ---------------- Gradio layout ----------------
def build_ui(client: ShopTalkClient, api_url: str, link_base: str = "", use_stream: bool = True) -> gr.Blocks:
    """Builds the Gradio Blocks app (not launched)."""
    with gr.Blocks(title="ShopTalk") as demo:
        gr.Markdown("# 🛍️ ShopTalk\nAsk for products in plain language. Follow-ups such as "
                    "*\"something similar but cheaper\"* use the previous results.")
        with gr.Row():
            status = gr.Markdown(health_banner(client.health(), api_url))
            refresh = gr.Button("Refresh status", size="sm", scale=0)
        state = gr.State(new_state)
        pending = gr.State("")
        with gr.Row(equal_height=False):
            with gr.Column(scale=5):
                chatbot = gr.Chatbot(label="Chat", height=520)
                with gr.Row():
                    msg = gr.Textbox(placeholder="e.g. Waterproof hiking boots, size 10", show_label=False,
                                     scale=5, max_lines=3, autofocus=True)
                    send = gr.Button("Send", variant="primary", scale=1)
                gr.Examples(EXAMPLES, inputs=msg, label="Try")
                latency = gr.Markdown("")
                with gr.Row():
                    up = gr.Button("👍 Helpful", size="sm")
                    down = gr.Button("👎 Not helpful", size="sm")
                    clear = gr.Button("🧹 New chat", size="sm")
                comment = gr.Textbox(placeholder="Optional comment for your rating", show_label=False, max_lines=2)
                fb_status = gr.Markdown("")
            with gr.Column(scale=6):
                gr.Markdown("### Products")
                products = gr.HTML(EMPTY_PRODUCTS)

        def _turn(message, chat, st):
            yield from chat_turn(client, message, chat, st, link_base, use_stream)

        for trigger in (msg.submit, send.click):
            trigger(add_user_message, [msg, chatbot], [msg, chatbot, pending], queue=False) \
                .then(_turn, [pending, chatbot, state], [chatbot, products, latency, state])
        up.click(lambda c, s: send_feedback(client, "up", c, s), [comment, state], [fb_status, comment, state])
        down.click(lambda c, s: send_feedback(client, "down", c, s), [comment, state], [fb_status, comment, state])
        clear.click(clear_chat, [state], [chatbot, products, latency, fb_status, state])
        refresh.click(lambda: health_banner(client.health(), api_url), None, status)
        demo.load(lambda: health_banner(client.health(), api_url), None, status)
    demo.queue(default_concurrency_limit=8)
    return demo


def create_server(api_url: str, use_stream: bool = True) -> FastAPI:
    """
    FastAPI app that serves the Gradio UI at / and the click-tracking redirect at /go/....

    /go/{request_id}/{product_id}?rank=N logs the click through the API (server side, so the API does
    not need to be public) and redirects the browser to the product page.
    """
    client = ShopTalkClient(api_url)
    server = FastAPI(title="ShopTalk UI")

    @server.get("/go/{request_id}/{product_id}")
    def go(request_id: str, product_id: str, rank: Optional[int] = None, session_id: Optional[str] = None):
        if not (_ID_RE.match(request_id) and _ID_RE.match(product_id)):
            return RedirectResponse("/", status_code=307)
        target = client.click(request_id, product_id, rank=rank, session_id=session_id)
        return RedirectResponse(target or PRODUCT_URL.format(id=product_id), status_code=307)

    @server.get("/ui-health")
    def ui_health():
        return {"ui": "ok", "api": client.health().get("status"), "api_url": api_url}

    demo = build_ui(client, api_url, link_base="", use_stream=use_stream)
    return gr.mount_gradio_app(server, demo, path="/", css=CSS, theme=gr.themes.Soft())


def main():
    import uvicorn
    api_url = os.environ.get("SHOPTALK_API_URL", "http://127.0.0.1:8000")
    port = int(os.environ.get("SHOPTALK_UI_PORT", "7860"))
    use_stream = os.environ.get("SHOPTALK_UI_STREAM", "1") != "0"
    uvicorn.run(create_server(api_url, use_stream), host="0.0.0.0", port=port, log_level="info")


if __name__ == "__main__":
    main()

### 📘 2.6 UI Tests (`test_ui.py`)

*   **Purpose:** 12 tests: SSE parsing across split chunks, HTML escaping, card links, the client endpoints, a streamed chat turn, a follow-up, no-match, non-streaming mode, feedback logged with the session id, the `/go` click redirect, and a friendly message when the API is down.
*   **Expected Results:** Writes `tests/test_ui.py`. Run in section 3.
*   **Parameters:** None.

In [ ]:
%%writefile tests/test_ui.py
"""Phase 8 UI tests: the Gradio handlers, the API client and the click redirect.

They start the Phase 7 API with the fake pipeline (SHOPTALK_FAKE=1: no GPU, no index) on a spare
port, so they run in seconds and check the UI against the real API contract.
"""
import json
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

import pytest
import requests

from src.ui import app as ui
from src.ui.client import ApiError, ShopTalkClient, parse_sse
from src.ui.render import format_answer, product_cards_html


def _free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


@pytest.fixture(scope="module")
def api(tmp_path_factory):
    """Fake-pipeline API in a subprocess. Yields (base_url, log_dir)."""
    log_dir = tmp_path_factory.mktemp("logs")
    port = _free_port()
    env = dict(os.environ, SHOPTALK_FAKE="1", SHOPTALK_LOG_DIR=str(log_dir),
               PYTHONPATH=os.environ.get("PYTHONPATH", os.getcwd()))
    proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "src.api.main:app", "--port", str(port)],
                            env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    base = f"http://127.0.0.1:{port}"
    try:
        for _ in range(120):
            try:
                if requests.get(f"{base}/health", timeout=2).json()["status"] == "ok":
                    break
            except requests.RequestException:
                pass
            time.sleep(0.5)
        else:
            pytest.fail("fake API did not start")
        yield base, Path(log_dir)
    finally:
        proc.terminate()
        proc.wait(timeout=10)


def _read(log_dir: Path, name: str) -> list:
    p = log_dir / f"{name}.jsonl"
    return [json.loads(l) for l in p.read_text().splitlines()] if p.exists() else []


def _run_turn(client, message, chat=None, state=None, use_stream=True):
    """Runs one chat turn the way Gradio does and returns every update plus the last one."""
    _, chat, pending = ui.add_user_message(message, chat)
    updates = list(ui.chat_turn(client, pending, chat, state, use_stream=use_stream))
    return updates, updates[-1]


# ---------------- pure functions ----------------
def test_parse_sse_handles_split_chunks():
    raw = b'event: meta\ndata: {"request_id": "r1"}\n\nevent: token\ndata: {"text": "Hel'
    raw2 = b'lo"}\n\nevent: done\ndata: {"answer": "Hello"}\n\n'
    events = list(parse_sse([raw[:10], raw[10:], raw2]))
    assert [e for e, _ in events] == ["meta", "token", "done"]
    assert events[1][1]["text"] == "Hello"


def test_cards_escape_html_and_link_through_go():
    html = product_cards_html([{"id": "B01", "title": "<script>x</script>", "price": 9.5, "link": "https://a/b"}],
                              request_id="rid1", session_id="s1")
    assert "<script>" not in html and "&lt;script&gt;" in html
    assert "/go/rid1/B01?rank=1&amp;session_id=s1" in html
    assert "$9.50" in html


def test_cards_no_match_and_answer_format():
    assert "No matching products" in product_cards_html([], no_match=True)
    assert format_answer("Try [ID: B082SP22KL].") == "Try **`B082SP22KL`**."
    assert "<b>" not in format_answer("<b>bold</b> [ID: X1]")


# ---------------- against the fake-pipeline API ----------------
def test_client_endpoints(api):
    base, log_dir = api
    c = ShopTalkClient(base)
    assert c.health()["status"] == "ok"
    b = c.recommend("red sneakers for men under $50", session_id="t-client")
    assert b["products"] and b["request_id"]
    kinds = [k for k, _ in c.stream("red sneakers for men", session_id="t-client")]
    assert kinds[0] == "meta" and "token" in kinds and kinds[-1] == "done"
    assert c.feedback(b["request_id"], "up", session_id="t-client")["status"] == "recorded"
    assert c.click(b["request_id"], b["products"][0]["id"], rank=1).startswith("https://")


def test_chat_turn_streams_products_then_answer(api):
    base, _ = api
    updates, (chat, products, latency, state) = _run_turn(ShopTalkClient(base), "red sneakers for men under $50")
    assert len(updates) >= 4                                   # searching, products, tokens..., final
    assert chat[0] == {"role": "user", "content": "red sneakers for men under $50"}
    assert chat[-1]["role"] == "assistant" and "▌" not in chat[-1]["content"]
    pid = state["last_response"]["products"][0]["id"]
    assert pid in products and "/go/" in products
    assert state["last_request_id"] and len(state["history"]) == 1
    assert latency.startswith("⏱️")


def test_follow_up_keeps_history(api):
    base, _ = api
    c = ShopTalkClient(base)
    _, (chat, _, _, state) = _run_turn(c, "sneakers for men")
    _, (chat, products, _, state) = _run_turn(c, "Show me something similar but cheaper", chat, state)
    assert len(state["history"]) == 2 and len(chat) == 4
    assert not chat[-1]["content"].startswith("⚠️")


def test_no_match(api):
    base, _ = api
    _, (chat, products, _, state) = _run_turn(ShopTalkClient(base), "a red shirt for men under $1")
    assert "No matching products" in products and state["last_response"]["no_match"]


def test_non_streaming_mode(api):
    base, _ = api
    _, (chat, products, _, state) = _run_turn(ShopTalkClient(base), "black leather wallet", use_stream=False)
    assert state["last_request_id"] and not chat[-1]["content"].startswith("⚠️")


def test_feedback_logged_with_session(api):
    base, log_dir = api
    c = ShopTalkClient(base)
    msg, _, _ = ui.send_feedback(c, "up", "", ui.new_state())
    assert "Ask something first" in msg
    _, (_, _, _, state) = _run_turn(c, "coffee gift")
    msg, comment, state = ui.send_feedback(c, "down", "not what I wanted", state)
    assert "recorded" in msg and comment == ""
    rec = [r for r in _read(log_dir, "feedback") if r.get("request_id") == state["last_request_id"]][-1]
    assert rec["rating"] == "down" and rec["session_id"] == state["session_id"]
    assert rec["comment"] == "not what I wanted"


def test_go_route_logs_click_and_redirects(api):
    from fastapi.testclient import TestClient
    base, log_dir = api
    _, (_, _, _, state) = _run_turn(ShopTalkClient(base), "red sneakers for men")
    rid, pid = state["last_request_id"], state["last_response"]["products"][0]["id"]
    tc = TestClient(ui.create_server(base))
    r = tc.get(f"/go/{rid}/{pid}?rank=1&session_id={state['session_id']}", follow_redirects=False)
    assert r.status_code == 307 and pid in r.headers["location"]
    clicks = [x for x in _read(log_dir, "feedback") if x.get("type") == "click" and x["request_id"] == rid]
    assert clicks and clicks[-1]["rank"] == 1 and clicks[-1]["session_id"] == state["session_id"]
    assert tc.get("/go/bad id!/x", follow_redirects=False).status_code == 307
    assert tc.get("/ui-health").json()["api"] == "ok"


def test_api_unreachable_shows_message():
    c = ShopTalkClient(f"http://127.0.0.1:{_free_port()}")
    assert c.health()["status"] == "unreachable"
    _, (chat, _, _, _) = _run_turn(c, "anything")
    assert chat[-1]["content"].startswith("⚠️")
    with pytest.raises(ApiError):
        c.recommend("x")


def test_ui_builds(api):
    base, _ = api
    demo = ui.build_ui(ShopTalkClient(base), base)
    assert demo is not None

### 📘 2.7 Library Version Pinning (`requirements-ui.txt`)

*   **Purpose:** Records the versions the UI ran with (plan, Phase 1: *pin library versions as you go*). Phase 11 builds the UI container from this file.
*   **Expected Results:** Writes and prints `requirements-ui.txt`.
*   **Parameters:** None.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
lines = []
for p in ["gradio", "fastapi", "uvicorn", "requests", "httpx"]:
    try:
        lines.append(f"{p}=={version(p)}")
    except PackageNotFoundError:
        lines.append(p)
(REPO_DIR / "requirements-ui.txt").write_text("\n".join(lines) + "\n")
print(open(REPO_DIR / "requirements-ui.txt").read())

## 3. Unit tests (fake API, no GPU)
The tests start the Phase 7 API with its fake pipeline on a spare port and drive the UI handlers against it, so they check the real API contract in seconds without touching the index or the LLM.

### 📘 3.1 Execution of UI Unit Tests

*   **Purpose:** Runs `tests/test_ui.py` with pytest in a subprocess. Test logs go to a temporary folder, not to `logs/` on Drive.
*   **Expected Results:** `12 passed`. Updates `CHECKS["UI unit tests pass"]`.
*   **Parameters:** None.

In [ ]:
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_ui.py"], cwd=REPO_DIR, capture_output=True, text=True,
                   env=dict(os.environ, PYTHONPATH=str(REPO_DIR), GRADIO_ANALYTICS_ENABLED="False"))
print(t.stdout[-3000:])
if t.returncode != 0:
    print(t.stderr[-3000:])
CHECKS["UI unit tests pass"] = t.returncode == 0
assert t.returncode == 0, "UI unit tests failed; see the output above"

## 4. Start the API
`local`: the real pipeline, exactly as in Phase 7 (1–3 min to load). `fake`: the fake pipeline (seconds). `remote`: nothing is started; the remote API is only checked.
Server output goes to `logs/server.log`.

### 📘 4.1 Launching (or Checking) the Phase 7 API

*   **Purpose:** Starts `uvicorn src.api.main:app` in the background (local / fake mode) and polls `/health` until it reports `ok`.
*   **Expected Results:** `✅ API ready: <LLM> | index <n> vectors`. Updates `CHECKS["API ready"]`.
*   **Parameters:** `API_MODE`, `API_PORT`, `LOCAL_CHROMA_DIR`.

In [ ]:
import requests

def wait_for_api(url, limit_s=600, proc=None):
    t0, h = time.time(), None
    while time.time() - t0 < limit_s:
        try:
            h = requests.get(f"{url}/health", timeout=5).json()
            if h["status"] in ("ok", "error"):
                break
        except requests.RequestException:
            pass
        if proc is not None and proc.poll() is not None:
            break
        time.sleep(5)
    return h, time.time() - t0

api_server = None
if API_MODE in ("local", "fake"):
    try:
        requests.get(f"{API_URL}/health", timeout=2)
        print(f"An API is already running on port {API_PORT}; reusing it (stop it in section 11 to restart).")
    except requests.RequestException:
        env = dict(os.environ, SHOPTALK_REPO_DIR=str(REPO_DIR), SHOPTALK_CONFIG=str(CONFIG_PATH),
                   SHOPTALK_LOCAL_CHROMA=LOCAL_CHROMA_DIR, SHOPTALK_LOG_DIR=str(LOG_DIR), PYTHONPATH=str(REPO_DIR))
        env.pop("SHOPTALK_FAKE", None)
        if API_MODE == "fake":
            env["SHOPTALK_FAKE"] = "1"
        api_server = subprocess.Popen([sys.executable, "-m", "uvicorn", "src.api.main:app", "--host", "127.0.0.1",
                                       "--port", str(API_PORT)], cwd=REPO_DIR, env=env,
                                      stdout=open(LOG_DIR / "server.log", "w"), stderr=subprocess.STDOUT)

h, waited = wait_for_api(API_URL, proc=api_server)
print(f"after {waited:.0f}s:", h)
if not h or h.get("status") != "ok":
    if (LOG_DIR / "server.log").exists():
        print(open(LOG_DIR / "server.log").read()[-3000:])
CHECKS["API ready"] = bool(h) and h.get("status") == "ok"
assert CHECKS["API ready"], "API not ready; see the output above"
print(f"✅ API ready: {h['llm']} | index {h['index_size']:,} vectors")

## 5. Start the UI server
Runs `python -m src.ui.app` in the background: Gradio at `/`, the click redirect at `/go/...`. Output goes to `logs/ui_server.log`.

### 📘 5.1 Launching the Gradio UI Server

*   **Purpose:** Starts the UI process pointed at `API_URL` and waits for `/ui-health`, which also reports whether the UI can reach the API.
*   **Expected Results:** `✅ UI ready on port 7860 (API: ok)`. Updates `CHECKS["UI server starts and reaches the API"]`.
*   **Parameters:** `UI_PORT`, `API_URL`.

In [ ]:
try:
    ui_server.terminate(); ui_server.wait(timeout=10)   # restart cleanly if this cell is re-run
except NameError:
    pass
ui_env = dict(os.environ, SHOPTALK_API_URL=API_URL, SHOPTALK_UI_PORT=str(UI_PORT), PYTHONPATH=str(REPO_DIR),
              GRADIO_ANALYTICS_ENABLED="False")
ui_server = subprocess.Popen([sys.executable, "-m", "src.ui.app"], cwd=REPO_DIR, env=ui_env,
                             stdout=open(LOG_DIR / "ui_server.log", "w"), stderr=subprocess.STDOUT)
ui_h = None
for _ in range(60):
    try:
        ui_h = requests.get(f"{UI_BASE}/ui-health", timeout=5).json()
        break
    except requests.RequestException:
        if ui_server.poll() is not None:
            break
        time.sleep(2)
print("UI health:", ui_h)
if not ui_h:
    print(open(LOG_DIR / "ui_server.log").read()[-3000:])
CHECKS["UI server starts and reaches the API"] = bool(ui_h) and ui_h.get("api") == "ok"
assert CHECKS["UI server starts and reaches the API"], "UI did not start or cannot reach the API; see above"
print(f"✅ UI ready on port {UI_PORT} (API: {ui_h['api']})")

## 6. Smoke tests through the UI
Runs the same handlers the browser triggers (`chat_turn`, `send_feedback`) against the live API, and the `/go` click route on the running UI server. These requests are logged with `session_id` `phase8-smoke` so they can be filtered out later.

### 📘 6.1 User Stories through the Chat Handler

*   **Purpose:** Sends the five Phase 7 user stories through `chat_turn` (streaming) and checks that each one ends with an answer and product cards, then reports how many cards have an image.
*   **Expected Results:** A table with each query, number of products, whether the answer cites product IDs, and the UI timings. Updates `CHECKS["user stories answered in the UI"]` and `CHECKS["product cards have images"]` (a warning, not a blocker).
*   **Parameters:** None.

In [ ]:
import pandas as pd
from src.ui import app as ui
from src.ui.client import ShopTalkClient

client = ShopTalkClient(API_URL)

def run_turn(message, chat=None, state=None, session_id="phase8-smoke"):
    state = state or ui.new_state(session_id)
    _, chat, pending = ui.add_user_message(message, chat)
    last = None
    for last in ui.chat_turn(client, pending, chat, state):
        pass
    return last   # (chat, products_html, latency_md, state)

story_turns = []
for q in ui.EXAMPLES:
    chat, products_html, latency_md, state = run_turn(q)
    r = state.get("last_response", {})
    story_turns.append({"query": q, "n_products": len(r.get("products", [])), "no_match": r.get("no_match"),
                        "cites_ids": any(f"`{p['id']}`" in chat[-1]["content"] for p in r.get("products", [])),
                        "error": chat[-1]["content"].startswith("⚠️"), **r.get("client_ms", {}),
                        "request_id": r.get("request_id")})
stories = pd.DataFrame(story_turns)
display(stories)
print("\nLast answer as shown in the chat:\n", chat[-1]["content"])

CHECKS["user stories answered in the UI"] = bool((~stories["error"]).all() and
                                                 (stories["no_match"] | (stories["n_products"] > 0)).all())

In [ ]:
# Image coverage: the cards fall back to a 🛍️ placeholder when image_url is missing
imgs = []
for q in ui.EXAMPLES[:3]:
    b = client.recommend(q, session_id="phase8-smoke")
    imgs += [bool(p.get("image_url")) for p in b["products"]]
share = sum(imgs) / max(1, len(imgs))
print(f"Cards with an image_url: {sum(imgs)}/{len(imgs)} ({share:.0%})")
CHECKS["product cards have images"] = True if share >= 0.5 else f"review: only {share:.0%} of cards have an image_url"

### 📘 6.2 Follow-Up and No-Match in the Chat

*   **Purpose:** Asks *sneakers for men*, then *Show me something similar but cheaper* in the same conversation (the handler sends back the stored `history_entry`), then an impossible request.
*   **Expected Results:** The follow-up shows cheaper products than the first turn's top result; the impossible request shows the no-match message and no cards. Updates `CHECKS["follow-up in the UI"]` and `CHECKS["no-match in the UI"]`.
*   **Parameters:** None.

In [ ]:
chat, _, _, state = run_turn("sneakers for men")
first_prices = [p["price"] for p in state["last_response"]["products"] if p.get("price") is not None]
chat, products_html, _, state = run_turn("Show me something similar but cheaper", chat, state)
fu_prices = [p["price"] for p in state["last_response"]["products"] if p.get("price") is not None]
for m in chat:
    print(f"{m['role']:>9}: {m['content'][:300]}")
print("\nfirst-turn prices:", first_prices, "\nfollow-up prices: ", fu_prices)
CHECKS["follow-up in the UI"] = (len(state["history"]) == 2 and bool(fu_prices) and bool(first_prices)
                                 and max(fu_prices) <= max(first_prices))

chat, products_html, _, nm_state = run_turn("a red shirt for men under $1")
print("\nNo-match answer:", chat[-1]["content"])
CHECKS["no-match in the UI"] = bool(nm_state["last_response"]["no_match"]) and "No matching products" in products_html

### 📘 6.3 Feedback and Click Tracking

*   **Purpose:** Sends a 👍 with a comment through `send_feedback`, opens a product card link through the running UI server's `/go` route, and checks both records reached `feedback.jsonl` with the session id.
*   **Expected Results:** `Thanks! 👍 recorded.`, a 307 redirect to the product page, and both records found in the log. Updates `CHECKS["feedback from the UI logged"]` and `CHECKS["clicks from the UI logged"]`.
*   **Parameters:** None.

In [ ]:
chat, products_html, _, state = run_turn("Find me a black leather wallet")
rid, sid = state["last_request_id"], state["session_id"]
msg, _, state = ui.send_feedback(client, "up", "phase 8 smoke test", state)
print(msg)

pid = state["last_response"]["products"][0]["id"]
r = requests.get(f"{UI_BASE}/go/{rid}/{pid}?rank=1&session_id={sid}", allow_redirects=False, timeout=10)
print("click ->", r.status_code, r.headers.get("location"))

if API_MODE == "remote":
    print("Remote API: its logs are on the other machine; checking only the HTTP responses.")
    CHECKS["feedback from the UI logged"] = msg.startswith("Thanks")
    CHECKS["clicks from the UI logged"] = r.status_code == 307
else:
    time.sleep(1)
    fb = pd.read_json(LOG_DIR / "feedback.jsonl", lines=True)
    mine = fb[fb["request_id"] == rid]
    display(mine)
    CHECKS["feedback from the UI logged"] = bool(((mine["type"] == "explicit") & (mine["rating"] == "up")
                                                  & (mine["session_id"] == sid)).any())
    CHECKS["clicks from the UI logged"] = r.status_code == 307 and bool(
        ((mine["type"] == "click") & (mine["product_id"] == pid) & (mine["rank"] == 1)).any())

## 7. UI latency
What the user waits for, measured at the UI server: until the product cards appear, until the first words of the answer, and until the answer is complete. Turns run one after another (no queueing), using the user stories repeated to `N_UI_LATENCY` turns.

### 📘 7.1 Time to Products, First Token and Full Answer

*   **Purpose:** Runs `N_UI_LATENCY` streamed chat turns and reports P50 / P95 / P99 of the three UI timings, next to Phase 7's API numbers.
*   **Expected Results:** Products in ≈0.3 s, first token ≈0.8 s and the full answer ≈3 s at P50 with the real pipeline (Phase 7: 263 ms / 774 ms / 4,662 ms P95 round trip). Saves `results/phase8/ui_latency_ms.csv`. Updates the two latency checks.
*   **Parameters:** `N_UI_LATENCY`, `UI_FIRST_TOKEN_P95_TARGET_MS`, `UI_TOTAL_P95_TARGET_MS`.

In [ ]:
qs = (ui.EXAMPLES * (N_UI_LATENCY // len(ui.EXAMPLES) + 1))[:N_UI_LATENCY]
rows = []
for q in qs:
    chat, _, _, state = run_turn(q, session_id="phase8-latency")
    r = state.get("last_response", {})
    rows.append({**r.get("client_ms", {}), "server_total_ms": (r.get("latency_ms") or {}).get("total"),
                 "no_match": r.get("no_match")})
lat = pd.DataFrame(rows)
cols = [c for c in ["products_ms", "first_token_ms", "total_ms", "server_total_ms"] if c in lat]
ui_latency = lat[cols].describe(percentiles=[.5, .95, .99]).loc[["mean", "50%", "95%", "99%", "max"]].round(0)
ui_latency.to_csv(RESULTS_DIR / "ui_latency_ms.csv")
display(ui_latency)

answered = lat[~lat["no_match"].astype(bool)]
p95_first = answered["first_token_ms"].quantile(.95)
p95_total = lat["total_ms"].quantile(.95)
print(f"First token P95: {p95_first:.0f} ms (target {UI_FIRST_TOKEN_P95_TARGET_MS}) | "
      f"full answer P95: {p95_total:.0f} ms (target {UI_TOTAL_P95_TARGET_MS})")
print(f"UI overhead (UI total - API pipeline total), median: {(lat['total_ms'] - lat['server_total_ms']).median():.0f} ms")
CHECKS["UI first-token P95 within target"] = bool(p95_first <= UI_FIRST_TOKEN_P95_TARGET_MS)
CHECKS["UI full-answer P95 within target"] = bool(p95_total <= UI_TOTAL_P95_TARGET_MS)

## 8. Public URL & manual test
Gives the **UI** a temporary public HTTPS address (the API stays private). It lasts while this Colab session runs and changes every time. Anyone with the URL can use the app and the GPU, so share it only when needed.

### 📘 8.1 Public Demo URL (Cloudflare Tunnel)

*   **Purpose:** Downloads `cloudflared` if needed and opens a tunnel to the UI port. If the tunnel is off or fails, prints a Colab-only link instead (works in your own browser while signed in to Colab).
*   **Expected Results:** `Public UI: https://….trycloudflare.com (reachable: True)`. Updates `CHECKS["public UI URL reachable"]`.
*   **Parameters:** `START_UI_TUNNEL`.

In [ ]:
import re
PUBLIC_UI_URL = None
if START_UI_TUNNEL:
    if not Path("/content/cloudflared").exists():
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
        !chmod +x /content/cloudflared
    try:
        ui_tunnel.terminate()
    except NameError:
        pass
    ui_tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{UI_PORT}", "--no-autoupdate"],
                                 stdout=open("/content/cloudflared_ui.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared_ui.log").read())
        if m:
            PUBLIC_UI_URL = m.group(0)
            break
    ok = False
    if PUBLIC_UI_URL:
        time.sleep(5)   # DNS for a new tunnel can take a few seconds
        for _ in range(10):
            try:
                ok = requests.get(f"{PUBLIC_UI_URL}/ui-health", timeout=10).json().get("ui") == "ok"
                break
            except Exception:
                time.sleep(3)
        print(f"Public UI: {PUBLIC_UI_URL}  (reachable: {ok})")
    else:
        print("Tunnel did not start; see /content/cloudflared_ui.log")
    CHECKS["public UI URL reachable"] = ok if PUBLIC_UI_URL else "review: tunnel did not start"
if not PUBLIC_UI_URL:
    from google.colab import output
    print("Colab-only link (opens in your browser while you are signed in to Colab):")
    output.serve_kernel_port_as_window(UI_PORT)

### 🧪 8.2 Manual test checklist (in the browser)

Open the URL from 8.1 (also try it on your phone) and go through these. Note anything odd for the write-up.

1. The status line at the top says 🟢 **API ready** with the LLM name and index size.
2. Click each **Try** example. Product cards appear first, then the answer streams in; cited IDs are highlighted and match cards on the right.
3. Ask *sneakers for men*, then *Show me something similar but cheaper*: the second set is cheaper and related to the first.
4. Ask something impossible (*a red shirt for men under $1*): a no-match message and no cards.
5. Click 👍 or 👎 (with and without a comment): a thanks message appears below.
6. Click **View product ↗** on a card: the Amazon page opens in a new tab.
7. Click **🧹 New chat**, then ask a follow-up: it is treated as a new question (no history).
8. Open a second tab and chat there: the two conversations do not mix.

Section 9 then shows your browser session's requests, ratings and clicks in the logs (sessions starting with `ui-`).

## 9. Logs (what the UI adds to the feedback loop)
UI requests, ratings and clicks go to the same `logs/requests.jsonl` and `logs/feedback.jsonl` as Phase 7, now with a `session_id` on every record. Browser sessions start with `ui-`; this notebook's test sessions with `phase8-`.

### 📘 9.1 Log Inspection by Session

*   **Purpose:** Reads both logs and summarizes this phase's records by session: requests, no-match answers, 👍, 👎 and clicks. Then joins ratings back to their queries.
*   **Expected Results:** A per-session table including your manual browser session(s) from 8.2. Updates `CHECKS["UI sessions in the logs"]`.
*   **Parameters:** None.

In [ ]:
if API_MODE == "remote":
    print("Remote API: the logs are written on the API's machine.")
    CHECKS["UI sessions in the logs"] = "review: remote API, logs not checked here"
else:
    req = pd.read_json(LOG_DIR / "requests.jsonl", lines=True)
    fb = pd.read_json(LOG_DIR / "feedback.jsonl", lines=True)
    is_p8 = lambda s: s.fillna("").str.startswith(("ui-", "phase8-"))
    req8, fb8 = req[is_p8(req["session_id"])], fb[is_p8(fb["session_id"])]
    aggs = {"requests": ("request_id", "count"), "first": ("ts", "min"), "last": ("ts", "max")}
    if "no_match" in req8:
        aggs["no_match"] = ("no_match", "sum")
    summary = req8.groupby("session_id").agg(**aggs)
    if len(fb8):
        counts = fb8.assign(kind=fb8["type"].where(fb8["type"] == "click", "rating_" + fb8["rating"].fillna(""))) \
                    .pivot_table(index="session_id", columns="kind", values="request_id", aggfunc="count", fill_value=0)
        summary = summary.join(counts, how="left").fillna(0)
    display(summary)
    rated = fb8[fb8["type"] == "explicit"].merge(req8[["request_id", "query", "product_ids"]], on="request_id", how="left")
    display(rated[["ts", "session_id", "rating", "comment", "query", "product_ids"]].tail(10))
    browser = [s for s in summary.index if s.startswith("ui-")]
    print("Browser sessions:", browser or "none yet. Do the manual test in 8.2, then rerun this cell.")
    CHECKS["UI sessions in the logs"] = True if browser else "review: no browser session yet (manual test 8.2)"

## 10. Phase 8 exit checklist

### 📘 10.1 Exit Checklist Evaluation

*   **Purpose:** Shows every check from this notebook. ✅ passed, ❌ blocking, ⚠️ to review (does not block).
*   **Expected Results:** `READY for Phase 9 (evaluation)` when nothing is ❌. Saves `results/phase8/phase8_checklist.csv`.
*   **Parameters:** None.

In [ ]:
for k, v in CHECKS.items():
    mark = "✅" if v is True else "❌" if v is False else "⚠️"
    print(f"{mark} {k}" + ("" if isinstance(v, bool) else f"  ({v})"))
blocking = [k for k, v in CHECKS.items() if v is False]
print("\n" + ("READY for Phase 9 (evaluation)" if not blocking else f"NOT READY: fix {blocking}"))
pd.Series({k: str(v) for k, v in CHECKS.items()}).to_csv(RESULTS_DIR / "phase8_checklist.csv")

## 11. Stop
Run this when you're done, to free the GPU and close the public URL. (Everything also stops when the Colab session ends.)

### 📘 11.1 Service Shutdown

*   **Purpose:** Stops the UI tunnel, the UI server, and the API if this notebook started it.
*   **Expected Results:** `stopped ui_tunnel`, `stopped ui_server`, `stopped api_server`.
*   **Parameters:** None.

In [ ]:
for name in ("ui_tunnel", "ui_server", "api_server"):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        print("stopped", name)

## 12. Write-up

### What to record
- A screenshot of the UI: a streamed answer with product cards, and a follow-up turn
- The UI flow (2.1): stream events → cards first, then tokens, then the cleaned final answer; `history_entry` for follow-ups; `/go` for clicks
- **UI latency table** (7.1): time to products, first token and full answer (P50 / P95 / P99), next to Phase 7's API numbers, and the UI overhead
- What the logs now hold (9.1): every request, rating and click carries a `session_id`, so a conversation can be followed end to end
- Results of the manual test (8.2), including anything that looked wrong (answer wording, missing images, slow turns)
- Known limits: one GPU serves one request at a time (concurrent users queue, Phase 7 section 6.2); the tunnel URL changes every session; no login, so anyone with the URL can use it

**Next, Phase 9 (evaluation):** run the evaluation queries through the API (and a sample through the UI), score retrieval and answers, and use the logs: thumbs-down and no-match requests become evaluation cases. Filter out test sessions (`phase7-…`, `phase8-…`, `test-…`) first.